# 🎯 Model Evaluation — License Plate Detector

Publication-quality evaluation of the trained YOLOv8s detector:
training curves, precision-recall, confusion matrix, F1-confidence,
sample predictions, OCR pipeline accuracy, and a speed benchmark.

**Requires:** `models/best.pt` (fine-tuned weights) and the dataset in `data/raw/`.
GPU recommended for the speed benchmark but not required.

### Setup

In [ ]:
import os
import random
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

sns.set_theme(style="whitegrid", palette="muted")

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

MODEL_PATH = PROJECT_ROOT / "models" / "best.pt"
RUN_DIR_CANDIDATES = sorted((PROJECT_ROOT / "runs").glob("**/license_plate"), key=os.path.getmtime)
RUN_DIR = RUN_DIR_CANDIDATES[-1] if RUN_DIR_CANDIDATES else None

cfg = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text())
DATA_YAML = PROJECT_ROOT / cfg["training"]["data_yaml"]
VIS_DIR = PROJECT_ROOT / cfg["output"]["visualization_dir"]
VIS_DIR.mkdir(parents=True, exist_ok=True)

assert MODEL_PATH.exists(), "models/best.pt not found — train first (notebooks/02_training.ipynb)"
print(f"Model     : {MODEL_PATH}")
print(f"Run dir   : {RUN_DIR}")
print(f"Dataset   : {DATA_YAML} (exists: {DATA_YAML.exists()})")

### 1 · Validate on the test split

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

if DATA_YAML.exists():
    metrics = model.val(data=str(DATA_YAML), split="test" if (DATA_YAML.parent / "test").exists() else "val")
    summary = {
        "mAP@50": round(float(metrics.box.map50), 3),
        "mAP@50-95": round(float(metrics.box.map), 3),
        "Precision": round(float(metrics.box.mp), 3),
        "Recall": round(float(metrics.box.mr), 3),
    }
else:
    metrics, summary = None, {}
    print("⚠️ Dataset config missing — running prediction-only checks.")

summary

### 2 · Training curves (from the training run)

In [ ]:
if RUN_DIR and (RUN_DIR / "results.csv").exists():
    res = pd.read_csv(RUN_DIR / "results.csv")
    res.columns = [c.strip() for c in res.columns]

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes[0, 0].plot(res["epoch"], res["train/box_loss"], label="train")
    axes[0, 0].plot(res["epoch"], res["val/box_loss"], label="val")
    axes[0, 0].set_title("Box loss"); axes[0, 0].legend()

    axes[0, 1].plot(res["epoch"], res["train/cls_loss"], label="train")
    axes[0, 1].plot(res["epoch"], res["val/cls_loss"], label="val")
    axes[0, 1].set_title("Classification loss"); axes[0, 1].legend()

    axes[1, 0].plot(res["epoch"], res["metrics/mAP50(B)"], color="#2ecc71", label="mAP@50")
    axes[1, 0].plot(res["epoch"], res["metrics/mAP50-95(B)"], color="#4a90d9", label="mAP@50-95")
    axes[1, 0].set_title("mAP curves"); axes[1, 0].legend()

    axes[1, 1].plot(res["epoch"], res["metrics/precision(B)"], color="#4a90d9", label="precision")
    axes[1, 1].plot(res["epoch"], res["metrics/recall(B)"], color="#2ecc71", label="recall")
    axes[1, 1].set_title("Precision / Recall"); axes[1, 1].legend()

    for ax in axes.flat:
        ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
    fig.suptitle("Training curves — YOLOv8s license plate detector", fontsize=14)
    fig.tight_layout()
    fig.savefig(VIS_DIR / "eval_training_curves.png", dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()
else:
    print("⚠️ No training run found under runs/ — train first to see curves.")

### 3 · Precision-Recall curve

Uses the curve saved by `model.val()` (Ultralytics writes `PR_curve.png` into the val run directory).

In [ ]:
val_dirs = sorted((PROJECT_ROOT / "runs").glob("**/val*"), key=os.path.getmtime) \
           + sorted((PROJECT_ROOT / "runs").glob("**/license_plate*"), key=os.path.getmtime)
pr_curve = next((d / "PR_curve.png" for d in reversed(val_dirs) if (d / "PR_curve.png").exists()), None)

if pr_curve:
    img = plt.imread(str(pr_curve))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Precision-Recall curve (from validation)", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(VIS_DIR / "eval_pr_curve.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
else:
    print("⚠️ No PR_curve.png found — run model.val() above first.")

### 4 · Detection confusion matrix (plate / no-plate)

Frame-level presence classification over the test images: an image counts as
**plate** if the ground truth has a box, and as **predicted** if the model
detects anything at the configured confidence.

In [ ]:
from sklearn.metrics import confusion_matrix

if DATA_YAML.exists():
    dcfg = yaml.safe_load(DATA_YAML.read_text())
    data_root = Path(dcfg.get("path", DATA_YAML.parent))
    if not data_root.is_absolute():
        data_root = (DATA_YAML.parent / data_root).resolve()
    test_rel = dcfg.get("test") or dcfg["val"]
    img_dir, lbl_dir = data_root / test_rel, (data_root / test_rel).parent / "labels"
    conf_threshold = cfg["detection"]["confidence_threshold"]

    imgs = sorted(img_dir.glob("*"))[:300]  # cap for runtime
    y_true, y_pred = [], []
    for p in imgs:
        gt = (lbl_dir / (p.stem + ".txt")).exists() and (lbl_dir / (p.stem + ".txt")).stat().st_size > 0
        dets = model.predict(str(p), conf=conf_threshold, verbose=False)
        pred = len(dets[0].boxes) > 0
        y_true.append(int(gt)); y_pred.append(int(pred))

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Plate", "Plate"],
                yticklabels=["No Plate", "Plate"],
                linewidths=0.5, linecolor="gray", ax=ax,
                annot_kws={"size": 14})
    ax.set_title("Detection Confusion Matrix", fontsize=14, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("Actual", fontsize=12)
    fig.tight_layout()
    fig.savefig(VIS_DIR / "eval_confusion_matrix.png", dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()
    tn, fp, fn, tp = cm.ravel()
    print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn} · "
          f"frame-level accuracy = {(tp + tn) / cm.sum():.3f} (n={len(y_true)})")
else:
    print("⚠️ Dataset not available — skipping confusion matrix.")

### 5 · F1–confidence curve

In [ ]:
f1_curve = next((d / "F1_curve.png" for d in reversed(val_dirs) if (d / "F1_curve.png").exists()), None)

if f1_curve:
    img = plt.imread(str(f1_curve))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("F1–confidence curve (from validation)", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(VIS_DIR / "eval_f1_curve.png", dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
else:
    print("⚠️ No F1_curve.png found — run model.val() above first.")

### 6 · Sample predictions (2×3 grid)

In [ ]:
if DATA_YAML.exists():
    samples = random.sample(sorted(img_dir.glob("*")), min(6, len(list(img_dir.glob("*")))))
    preds = model.predict([str(p) for p in samples], conf=conf_threshold, verbose=False)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, r in zip(axes.flat, preds):
        img = cv2.cvtColor(r.orig_img, cv2.COLOR_BGR2RGB)
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(img, (x1, y1), (x2, y2), (46, 204, 113), 2)
            cv2.putText(img, f"{float(box.conf[0]):.2f}", (x1, max(0, y1 - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (46, 204, 113), 2)
        ax.imshow(img)
        ax.axis("off")
    fig.suptitle(f"Predictions at conf={conf_threshold}", fontsize=14)
    fig.tight_layout()
    fig.savefig(VIS_DIR / "eval_sample_predictions.png", dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()
else:
    print("⚠️ Dataset not available — skipping prediction grid.")

### 7 · OCR accuracy on detected plates

The Kaggle detection dataset has **no ground-truth plate text**, so full OCR
accuracy can't be computed against labels. This cell reports the share of
detected plates where the OCR + Indian-plate validation chain produces a
plausible plate string — a useful sanity signal until text-labeled data
(Level 2) exists.

In [ ]:
if DATA_YAML.exists():
    from src.ocr import PlateOCR
    from src.postprocess import clean_plate_text, validate_plate
    from src.preprocess import preprocess_plate

    ocr = PlateOCR(cfg)
    total, plausible = 0, 0
    for p in random.sample(sorted(img_dir.glob("*")), min(100, len(list(img_dir.glob("*"))))):
        for det in model.predict(str(p), conf=conf_threshold, verbose=False):
            for box in det.boxes.xyxy.int().tolist():
                x1, y1, x2, y2 = box
                crop = det.orig_img[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                text, _conf = ocr.recognize(preprocess_plate(crop, cfg))
                cleaned = clean_plate_text(text)
                total += 1
                plausible += int(validate_plate(cleaned))

    if total:
        print(f"OCR chain: {plausible}/{total} cropped plates "
              f"({100 * plausible / total:.1f}%) produced a format-valid Indian plate string")
    else:
        print("No detections found in the sampled images.")
else:
    print("⚠️ Dataset not available — skipping OCR accuracy.")

### 8 · Speed benchmark (detection + full OCR chain)

In [ ]:
bench = sorted(img_dir.glob("*"))[:50] if DATA_YAML.exists() else []
if bench:
    from src.ocr import PlateOCR
    from src.preprocess import preprocess_plate

    ocr = PlateOCR(cfg)
    t_detect, t_ocr = 0.0, 0.0
    paths = [str(p) for p in bench]
    for _ in range(3):  # warmup
        model.predict(paths[0], verbose=False)

    t0 = time.perf_counter()
    for p in paths:
        dets = model.predict(p, conf=conf_threshold, verbose=False)
    t_detect = (time.perf_counter() - t0) / len(paths)

    crops = [d.orig_img[0:60, 0:200] for d in dets[:1]] * min(len(paths), 20) or [np.zeros((60, 200, 3), np.uint8)] * 20
    t0 = time.perf_counter()
    for c in crops:
        ocr.recognize(preprocess_plate(c, cfg))
    t_ocr = (time.perf_counter() - t0) / len(crops)

    fig, ax = plt.subplots(figsize=(8, 4))
    parts = ["YOLO detection", "Preprocess+OCR"]
    vals = [t_detect * 1000, t_ocr * 1000]
    lefts = [0, t_detect * 1000]
    ax.barh(parts, vals, left=lefts, color=["#4a90d9", "#2ecc71"], height=0.5)
    for p_, v, l in zip(parts, vals, lefts):
        ax.text(l + v / 2, p_, f"{v:.1f} ms", va="center", ha="center", color="white", fontweight="bold")
    ax.set_title(f"Per-frame latency breakdown (target ≤ 150 ms)")
    ax.set_xlabel("milliseconds")
    fig.tight_layout()
    fig.savefig(VIS_DIR / "eval_latency_breakdown.png", dpi=300,
                bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Detection: {t_detect * 1000:.1f} ms · OCR chain: {t_ocr * 1000:.1f} ms · "
          f"total: {(t_detect + t_ocr) * 1000:.1f} ms · approx FPS: {1 / (t_detect + t_ocr):.1f}")
else:
    print("⚠️ Dataset not available — skipping speed benchmark.")

## 📝 Results summary

| Metric | Value | Target (Level 1) |
|--------|-------|------------------|
| mAP@50 | see §1 | ≥ 0.85 |
| mAP@50-95 | see §1 | — |
| Frame-level detection accuracy | see §4 | — |
| OCR chain plausibility | see §7 | — |
| End-to-end latency | see §8 | ≤ 150 ms |

**Recommendations**
- If mAP@50 ≥ 0.85 → ship as Level 1 weights.
- If recall lags precision, lower `detection.confidence_threshold` in `config/config.yaml`.
- If latency > 150 ms, ensure YOLO runs on GPU and EasyOCR stays on CPU (6GB VRAM budget).
- Charts are saved to `outputs/visualizations/` at 300 DPI for the README/report.